In [1]:
# %matplotlib ipympl

In [1]:
import contextlib
import itertools
import multiprocessing as mp
from pathlib import Path

import numpy as np
import pandas as pd
import pyPPG
import pyPPG.example
import scipy.io
import scipy.signal
from matplotlib import pyplot as plt
from tqdm import notebook, tqdm

In [5]:
csvs_dir = Path('data/csvs')

In [3]:
ppg_dir = Path('data/ppg')

In [4]:
pyppg_dir = Path('data/pyppg')

In [6]:
df_subjects = pd.read_csv(csvs_dir / 'subjects.csv')

In [7]:
df_subjects['patient_file'] = df_subjects.apply(lambda r: ppg_dir / f'p{r["subject_id"]:06d}_{r["segment_id"]}.mat', axis=1)

In [8]:
fs = 125

In [9]:
def make_filter(design: str, wn: int):
    if design == 'cheby2':
        filt = scipy.signal.cheby2(4, 20, wn, btype='band' if isinstance(wn, list) else 'low', fs=fs, output='sos')
    elif design == 'butter':
        filt = scipy.signal.butter(4, wn, btype='band' if isinstance(wn, list) else 'low', fs=fs, output='sos')
    elif design == 'bessel':
        filt = scipy.signal.bessel(4, wn, btype='band' if isinstance(wn, list) else 'low', fs=fs, output='sos')
    else:
        raise NotImplementedError
    return filt


designs = ['cheby2', 'butter', 'bessel']

# wns = [10, 15, 20, 25, 30, 35, [0.5, 10], [0.5, 12], [0.5, 15], [0.5, 20], [0.5, 25], [0.5, 30], [0.5, 35], [0.1, 30]]
wns = [[0.1, 30], [0.5, 12]]

In [10]:
df_iter = df_subjects[['subject_id', 'class', 'patient_file']].itertuples(index=False)

for (subject_id, category, patient_file), design, wn in tqdm(list(itertools.product(df_iter, designs, wns))):

    mat = scipy.io.loadmat(patient_file)

    # t = mat['time'].squeeze()
    ppg_raw = mat['data'].squeeze()

    t0 = 2 * 60 * fs
    t1 = t0 + 1 * 60 * fs
    ppg_raw = ppg_raw[t0:t1]

    filt = make_filter(design, wn)
    ppg_filt = scipy.signal.sosfiltfilt(filt, ppg_raw)

    signal_file = f'{patient_file.stem}_{design}_{wn}.mat'

    scipy.io.savemat((pyppg_dir / '2+1' / signal_file).as_posix(), {'Fs': fs, 'Data': ppg_filt})

100%|██████████| 204/204 [00:00<00:00, 223.26it/s]


In [ ]:
data = {
    'fn': []
    'mean'
}

df_iter = df_subjects.loc[:1, ['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects.sample(3, random_state=0)[['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects[['subject_id', 'class', 'patient_file']].itertuples(index=False)

for (subject_id, category, patient_file), design, wn in tqdm(list(itertools.product(df_iter, designs, wns))):

    signal_file = f'{patient_file.stem}_{design}_{wn}.mat'

    # with contextlib.redirect_stdout(None), contextlib.redirect_stderr(None):
    with contextlib.redirect_stderr(None):
        s, fp, bm = pyPPG.example.ppg_example(
            data_path=(pyppg_dir / '2+1' / signal_file).as_posix(),
            process_type='fiducials',
            filtering=False,
            # filtering=True,
            # fL=0,
            # fH=30,
            print_flag=False,
            plotfig=False,
            savefig=False,
            savedata=False
        )

    nas = fp.get_fp().isna().sum()
    nas.name = (str(category), str(subject_id), str(design), str(wn))
    data_nas.append(nas)

df_nas = pd.DataFrame(data_nas)
df_nas.index = df_nas.index.set_names(['category', 'subject_id', 'design', 'wn'])
df_nas

In [37]:
data_nas = []

df_iter = df_subjects.loc[:1, ['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects.sample(3, random_state=0)[['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects[['subject_id', 'class', 'patient_file']].itertuples(index=False)

for (subject_id, category, patient_file), design, wn in tqdm(list(itertools.product(df_iter, designs, wns))):

    signal_file = f'{patient_file.stem}_{design}_{wn}.mat'

    # with contextlib.redirect_stdout(None), contextlib.redirect_stderr(None):
    with contextlib.redirect_stderr(None):
        s, fp, bm = pyPPG.example.ppg_example(
            data_path=(pyppg_dir / '2+1' / signal_file).as_posix(),
            process_type='fiducials',
            filtering=False,
            # filtering=True,
            # fL=0,
            # fH=30,
            print_flag=False,
            plotfig=False,
            savefig=False,
            savedata=False
        )

    nas = fp.get_fp().isna().sum()
    nas.name = (str(category), str(subject_id), str(design), str(wn))
    data_nas.append(nas)

df_nas = pd.DataFrame(data_nas)
df_nas.index = df_nas.index.set_names(['category', 'subject_id', 'design', 'wn'])
df_nas

100%|██████████| 12/12 [00:02<00:00,  5.37it/s]


on  sp  dn  dp  off  u  v  w  a  b  c  \
category subject_id design wn                                                 
control  48480      cheby2 [0.1, 30]   0   0   0   7    0  0  2  2  0  0  4   
                           [0.5, 12]   0   0   0  58    0  0  0  0  0  0  4   
                    butter [0.1, 30]   0   0   0   3    0  0  2  2  0  0  4   
                           [0.5, 12]   0   0   0  13    0  0  2  2  0  0  6   
                    bessel [0.1, 30]   0   0   0  12    0  0  0  0  0  0  4   
                           [0.5, 12]   0   0   0  39    0  0  0  0  0  0  0   
         63646      cheby2 [0.1, 30]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 12]   0   0   0  14    0  0  0  1  0  0  0   
                    butter [0.1, 30]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 12]   0   0   0   7    0  0  0  2  0  0  0   
                    bessel [0.1, 30]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 12]   0   0   0  15    0  0  0  1  0  0  0   

                                      d  e  f  p1  p2  
category subject_id design wn                          
control  48480      cheby2 [0.1, 30]  4  0  4   0   4  
                           [0.5, 12]  4  0  4   0   4  
                    butter [0.1, 30]  4  0  4   0   4  
                           [0.5, 12]  6  0  6   0   6  
                    bessel [0.1, 30]  4  0  4   0   4  
                           [0.5, 12]  0  0  0   0   0  
         63646      cheby2 [0.1, 30]  0  0  0   0   0  
                           [0.5, 12]  0  0  0   0   0  
                    butter [0.1, 30]  0  0  0   0   0  
                           [0.5, 12]  0  0  0   0   0  
                    bessel [0.1, 30]  0  0  0   0   0  
                           [0.5, 12]  0  0  0   0   0

In [ ]:
data_means_y = []

df_iter = df_subjects.loc[:1, ['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects.sample(3, random_state=0)[['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects[['subject_id', 'class', 'patient_file']].itertuples(index=False)

for (subject_id, category, patient_file), design, wn in tqdm(list(itertools.product(df_iter, designs, wns))):

    signal_file = f'{patient_file.stem}_{design}_{wn}.mat'

    # with contextlib.redirect_stdout(None), contextlib.redirect_stderr(None):
    with contextlib.redirect_stderr(None):
        s, fp, bm = pyPPG.example.ppg_example(
            data_path=(pyppg_dir / '2+1' / signal_file).as_posix(),
            process_type='fiducials',
            filtering=False,
            # filtering=True,
            # fL=0,
            # fH=30,
            print_flag=False,
            plotfig=False,
            savefig=False,
            savedata=False
        )

    mean = fp.get_fp().map(lambda x: s.v[int(x)], na_action='ignore').mean()
    mean.name = (str(category), str(subject_id), str(design), str(wn))
    data_means_y.append(mean)

df_mean = pd.DataFrame(data_means_y)
df_mean.index = df_mean.index.set_names(['category', 'subject_id', 'design', 'wn'])
df_mean

100%|██████████| 12/12 [00:02<00:00,  5.15it/s]


on        sp        dn        dp  \
category subject_id design wn                                                  
control  48480      cheby2 [0.1, 30] -0.132868  0.275634 -0.090691  0.027968   
                           [0.5, 12] -0.135746  0.273087 -0.095396  0.002653   
                    butter [0.1, 30] -0.134978  0.272976 -0.091897  0.023158   
                           [0.5, 12] -0.139171  0.269633 -0.097497 -0.013625   
                    bessel [0.1, 30] -0.134245  0.271632 -0.094083 -0.008679   
                           [0.5, 12] -0.133699  0.255007 -0.100431 -0.089326   
         63646      cheby2 [0.1, 30] -0.585530  0.926957  0.034805 -0.045950   
                           [0.5, 12] -0.595208  0.918368  0.062916 -0.171462   
                    butter [0.1, 30] -0.619791  0.892948  0.001723 -0.035211   
                           [0.5, 12] -0.610987  0.900742  0.031416 -0.156219   
                    bessel [0.1, 30] -0.616439  0.893802  0.004333 -0.115689   
                           [0.5, 12] -0.588850  0.859916  0.012188 -0.203890   

                                           off         u         v         w  \
category subject_id design wn                                                  
control  48480      cheby2 [0.1, 30] -0.134650  0.059322  0.172584 -0.012753   
                           [0.5, 12] -0.137105  0.073411  0.164257 -0.079296   
                    butter [0.1, 30] -0.136874  0.059093  0.168148 -0.021756   
                           [0.5, 12] -0.140002  0.057895  0.160564 -0.062226   
                    bessel [0.1, 30] -0.135586  0.061574  0.163802 -0.020215   
                           [0.5, 12] -0.134545  0.064566  0.139962 -0.096688   
         63646      cheby2 [0.1, 30] -0.584892  0.089693  0.597894  0.114211   
                           [0.5, 12] -0.601135  0.101328  0.579550 -0.097933   
                    butter [0.1, 30] -0.618398  0.045009  0.562741  0.117580   
                           [0.5, 12] -0.616237  0.074463  0.567282 -0.046900   
                    bessel [0.1, 30] -0.617824  0.063959  0.563036  0.075363   
                           [0.5, 12] -0.594533  0.092018  0.523537 -0.149753   

                                             a         b         c         d  \
category subject_id design wn                                                  
control  48480      cheby2 [0.1, 30] -0.122386  0.142055  0.143003  0.177188   
                           [0.5, 12] -0.125576  0.241918  0.220588  0.216324   
                    butter [0.1, 30] -0.122944  0.094193  0.130913  0.226946   
                           [0.5, 12] -0.129335  0.228296  0.162336  0.151221   
                    bessel [0.1, 30] -0.124120  0.187969  0.157419  0.162303   
                           [0.5, 12] -0.118381  0.228902  0.226946  0.226055   
         63646      cheby2 [0.1, 30] -0.495267  0.094705  0.311668  0.771196   
                           [0.5, 12] -0.526477  0.618739  0.870201  0.877656   
                    butter [0.1, 30] -0.518515  0.070176  0.287100  0.743111   
                           [0.5, 12] -0.537641  0.582588  0.814640  0.837602   
                    bessel [0.1, 30] -0.530081  0.105549  0.302911  0.768630   
                           [0.5, 12] -0.519511  0.657485  0.740596  0.746811   

                                             e         f        p1        p2  
category subject_id design wn                                                 
control  48480      cheby2 [0.1, 30] -0.022665  0.007103  0.177622  0.181291  
                           [0.5, 12] -0.043210 -0.072071  0.264022 -0.057845  
                    butter [0.1, 30] -0.023493 -0.007598  0.127824  0.183650  
                           [0.5, 12] -0.033453 -0.049030  0.260558  0.062564  
                    bessel [0.1, 30] -0.013962 -0.004191  0.224426  0.181053  
                           [0.5, 12] -0.028177 -0.095625  0.231764 -0.081034  
         63646      cheby2 [0.1, 30]  0.270745  0

In [67]:
data_dists = []

df_iter = df_subjects.loc[:1, ['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects.sample(3, random_state=0)[['subject_id', 'class', 'patient_file']].itertuples(index=False)
# df_iter = df_subjects[['subject_id', 'class', 'patient_file']].itertuples(index=False)

for (subject_id, category, patient_file), design, wn in tqdm(list(itertools.product(df_iter, designs, wns))):

    signal_file = f'{patient_file.stem}_{design}_{wn}.mat'

    # with contextlib.redirect_stdout(None), contextlib.redirect_stderr(None):
    with contextlib.redirect_stderr(None):
        s, fp, bm = pyPPG.example.ppg_example(
            data_path=(pyppg_dir / '2+1' / signal_file).as_posix(),
            process_type='fiducials',
            filtering=False,
            # filtering=True,
            # fL=0,
            # fH=30,
            print_flag=False,
            plotfig=False,
            savefig=False,
            savedata=False
        )

    points = fp.get_fp()
    # points.loc[:, 'on':'off'] = (points.loc[:, 'on':'off'].T - points['on']).T
    # points.loc[:, 'u':'w'] = (points.loc[:, 'u':'w'].T - points['u']).T
    # points.loc[:, 'a':'f'] = (points.loc[:, 'a':'f'].T - points['a']).T
    # points.loc[:, 'p1':'p2'] = (points.loc[:, 'p1':'p2'].T - points['p1']).T
    points = (points.T - points['on']).T
    points = points.mean()
    points.name = (str(category), str(subject_id), str(design), str(wn))
    data_dists.append(points)

df_dists = pd.DataFrame(data_dists)
df_dists.index = df_dists.index.set_names(['category', 'subject_id', 'design', 'wn'])
df_dists

100%|██████████| 12/12 [00:02<00:00,  5.21it/s]


on         sp         dn         dp  \
category subject_id design wn                                                
control  48480      cheby2 [0.1, 30]  0.0  17.964286  50.892857  55.207792   
                           [0.5, 12]  0.0  16.815534  46.757282  53.200000   
                    butter [0.1, 30]  0.0  17.223529  50.988235  54.951220   
                           [0.5, 12]  0.0  16.925532  47.851064  53.395062   
                    bessel [0.1, 30]  0.0  17.031915  48.021277  52.865854   
                           [0.5, 12]  0.0  17.775862  43.965517  46.259740   
         63646      cheby2 [0.1, 30]  0.0  24.627907  45.651163  49.127907   
                           [0.5, 12]  0.0  24.523256  44.348837  53.444444   
                    butter [0.1, 30]  0.0  24.593023  45.604651  47.651163   
                           [0.5, 12]  0.0  24.581395  44.790698  52.037975   
                    bessel [0.1, 30]  0.0  24.802326  45.732558  50.720930   
                           [0.5, 12]  0.0  24.779070  45.186047  55.239437   

                                            off          u          v  \
category subject_id design wn                                           
control  48480      cheby2 [0.1, 30]  85.571429  17.940476  30.219512   
                           [0.5, 12]  69.359223  13.398058  27.650485   
                    butter [0.1, 30]  84.564706  16.447059  28.771084   
                           [0.5, 12]  76.457447  16.893617  29.445652   
                    bessel [0.1, 30]  76.468085  14.297872  28.925532   
                           [0.5, 12]  62.129310   9.689655  25.043103   
         63646      cheby2 [0.1, 30]  82.895349  11.500000  33.895349   
                           [0.5, 12]  82.895349  11.616279  34.000000   
                    butter [0.1, 30]  82.895349  11.383721  33.860465   
                           [0.5, 12]  82.895349  11.511628  33.918605   
                    bessel [0.1, 30]  82.895349  11.697674  34.069767   
                           [0.5, 12]  82.895349  12.162791  34.360465   

                                              w         a          b  \
category subject_id design wn                                          
control  48480      cheby2 [0.1, 30]  44.817073  7.190476  14.821429   
                           [0.5, 12]  49.174757  5.291262  16.291262   
                    butter [0.1, 30]  42.879518  6.576471  12.764706   
                           [0.5, 12]  48.478261  7.180851  17.457447   
                    bessel [0.1, 30]  41.829787  5.893617  14.957447   
                           [0.5, 12]  45.629310  3.379310  14.939655   
         63646      cheby2 [0.1, 30]  43.686047  5.151163  11.302326   
                           [0.5, 12]  50.588235  4.627907  16.616279   
                    butter [0.1, 30]  42.674419  5.302326  11.406977   
                           [0.5, 12]  47.714286  4.813953  16.523256   
                    bessel [0.1, 30]  44.081395  5.197674  11.860465   
                           [0.5, 12]  52.341176  4.790698  18.465116   

                                              c          d          e  \
category subject_id design wn                                           
control  48480      cheby2 [0.1, 30]  17.837500  23.462500  40.130952   
                           [0.5, 12]  20.575758  22.161616  35.893204   
                    butter [0.1, 30]  12.987654  17.074074  39.929412   
                           [0.5, 12]  23.136364  26.193182  37.351064   
                    bessel [0.1, 30]  19.766667  23.966667  36.904255   
                           [0.5, 12]  20.801724  21.000000  32.853448   
         63646      cheby2 [0.1, 30]  13.906977  19.941860  39.709302   
                           [0.5, 12]  24.430233  26.302326  39.779070   
                    butter [0.1, 30]  13.953488  20.023256  39.290698   
                           [0.5, 12]  23.674419  26.383721  40.244186   
                    bessel [0.1, 30]  

In [13]:
df.to_csv('data/pyppg/2+1/results.csv')

In [10]:
df = pd.read_csv('data/pyppg/2+1/results.csv', index_col=['category', 'subject_id', 'design', 'wn'])
df

on  sp  dn  dp  off  u  v  w  a  b  c  \
category subject_id design wn                                                 
control  48480      cheby2 10          0   0   0  99    0  0  0  0  0  0  1   
                           15          0   0   0  11    0  0  0  0  0  0  5   
                           20          0   0   0  18    0  0  3  3  0  0  4   
                           25          0   0   0  17    0  0  3  3  0  0  2   
                           30          0   0   0   7    0  0  2  2  0  0  2   
...                                   ..  ..  ..  ..  ... .. .. .. .. .. ..   
diab     98484      bessel [0.5, 20]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 25]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 30]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.5, 35]   0   0   0   0    0  0  0  0  0  0  0   
                           [0.1, 30]   0   0   0   1    0  0  0  0  0  0  0   

                                      d  e  f  p1  p2  
category subject_id design wn                          
control  48480      cheby2 10         1  0  1   0   1  
                           15         5  0  5   0   5  
                           20         4  0  4   0   4  
                           25         2  0  2   0   2  
                           30         2  0  2   0   2  
...                                  .. .. ..  ..  ..  
diab     98484      bessel [0.5, 20]  0  0  0   0   0  
                           [0.5, 25]  0  0  0   0   0  
                           [0.5, 30]  0  0  0   0   0  
                           [0.5, 35]  0  0  0   0   0  
                           [0.1, 30]  0  0  0   0   0  

[1428 rows x 16 columns]

In [15]:
df.groupby(level='design').mean()

,on,sp,dn,dp,off,u,v,w,a,b,c,d,e,f,p1,p2
design,,,,,,,,,,,,,,,,
bessel,0.0,0.0,0.0,4.640756,0.0,0.0,0.407563,0.834034,0.0,0.0,0.432773,0.432773,0.0,0.432773,0.0,0.432773
butter,0.0,0.0,0.0,2.750000,0.0,0.0,0.466387,0.544118,0.0,0.0,0.460084,0.460084,0.0,0.460084,0.0,0.460084
cheby2,0.0,0.0,0.0,3.964286,0.0,0.0,0.426471,0.512605,0.0,0.0,0.470588,0.470588,0.0,0.470588,0.0,0.470588


In [16]:
df.groupby(level='design').sum()

,on,sp,dn,dp,off,u,v,w,a,b,c,d,e,f,p1,p2
design,,,,,,,,,,,,,,,,
bessel,0,0,0,2209,0,0,194,397,0,0,206,206,0,206,0,206
butter,0,0,0,1309,0,0,222,259,0,0,219,219,0,219,0,219
cheby2,0,0,0,1887,0,0,203,244,0,0,224,224,0,224,0,224


In [17]:
df.groupby(level='wn').mean()

,on,sp,dn,dp,off,u,v,w,a,b,c,d,e,f,p1,p2
wn,,,,,,,,,,,,,,,,
10,0.0,0.0,0.0,11.666667,0.0,0.0,0.147059,1.735294,0.0,0.0,0.196078,0.196078,0.0,0.196078,0.0,0.196078
15,0.0,0.0,0.0,4.588235,0.0,0.0,0.362745,0.539216,0.0,0.0,0.529412,0.529412,0.0,0.529412,0.0,0.529412
20,0.0,0.0,0.0,3.196078,0.0,0.0,0.490196,0.568627,0.0,0.0,0.509804,0.509804,0.0,0.509804,0.0,0.509804
25,0.0,0.0,0.0,2.352941,0.0,0.0,0.558824,0.627451,0.0,0.0,0.460784,0.460784,0.0,0.460784,0.0,0.460784
30,0.0,0.0,0.0,1.676471,0.0,0.0,0.637255,0.686275,0.0,0.0,0.470588,0.470588,0.0,0.470588,0.0,0.470588
35,0.0,0.0,0.0,1.215686,0.0,0.0,0.617647,0.666667,0.0,0.0,0.460784,0.460784,0.0,0.460784,0.0,0.460784
"[0.1, 30]",0.0,0.0,0.0,1.774510,0.0,0.0,0.598039,0.656863,0.0,0.0,0.411765,0.411765,0.0,0.411765,0.0,0.411765
"[0.5, 10]",0.0,0.0,0.0,8.568627,0.0,0.0,0.039216,0.352941,0.0,0.0,0.137255,0.137255,0.0,0.137255,0.0,0.137255
"[0.5, 12]",0.0,0.0,0.0,5.843137,0.0,0.0,0.196078,0.372549,0.0,0.0,0.333333,0.333333,0.0,0.333333,0.0,0.333333


In [20]:
df.groupby(level='wn').sum()

,on,sp,dn,dp,off,u,v,w,a,b,c,d,e,f,p1,p2
wn,,,,,,,,,,,,,,,,
10,0,0,0,1190,0,0,15,177,0,0,20,20,0,20,0,20
15,0,0,0,468,0,0,37,55,0,0,54,54,0,54,0,54
20,0,0,0,326,0,0,50,58,0,0,52,52,0,52,0,52
25,0,0,0,240,0,0,57,64,0,0,47,47,0,47,0,47
30,0,0,0,171,0,0,65,70,0,0,48,48,0,48,0,48
35,0,0,0,124,0,0,63,68,0,0,47,47,0,47,0,47
"[0.1, 30]",0,0,0,181,0,0,61,67,0,0,42,42,0,42,0,42
"[0.5, 10]",0,0,0,874,0,0,4,36,0,0,14,14,0,14,0,14
"[0.5, 12]",0,0,0,596,0,0,20,38,0,0,34,34,0,34,0,34


In [21]:
df.groupby(level=['design', 'wn']).mean()

on   sp   dn         dp  off    u         v         w    a  \
design wn                                                                       
bessel 10         0.0  0.0  0.0  17.735294  0.0  0.0  0.264706  4.705882  0.0   
       15         0.0  0.0  0.0   6.352941  0.0  0.0  0.205882  0.588235  0.0   
       20         0.0  0.0  0.0   3.823529  0.0  0.0  0.264706  0.411765  0.0   
       25         0.0  0.0  0.0   2.735294  0.0  0.0  0.382353  0.441176  0.0   
       30         0.0  0.0  0.0   1.911765  0.0  0.0  0.500000  0.558824  0.0   
       35         0.0  0.0  0.0   1.500000  0.0  0.0  0.558824  0.647059  0.0   
       [0.1, 30]  0.0  0.0  0.0   2.147059  0.0  0.0  0.441176  0.529412  0.0   
       [0.5, 10]  0.0  0.0  0.0   9.176471  0.0  0.0  0.000000  0.470588  0.0   
       [0.5, 12]  0.0  0.0  0.0   6.176471  0.0  0.0  0.000000  0.205882  0.0   
       [0.5, 15]  0.0  0.0  0.0   4.323529  0.0  0.0  0.088235  0.117647  0.0   
       [0.5, 20]  0.0  0.0  0.0   3.411765  0.0  0.0  0.352941  0.352941  0.0   
       [0.5, 25]  0.0  0.0  0.0   2.411765  0.0  0.0  0.735294  0.735294  0.0   
       [0.5, 30]  0.0  0.0  0.0   1.823529  0.0  0.0  0.882353  0.882353  0.0   
       [0.5, 35]  0.0  0.0  0.0   1.441176  0.0  0.0  1.029412  1.029412  0.0   
butter 10         0.0  0.0  0.0   6.441176  0.0  0.0  0.117647  0.323529  0.0   
       15         0.0  0.0  0.0   3.529412  0.0  0.0  0.500000  0.529412  0.0   
       20         0.0  0.0  0.0   2.323529  0.0  0.0  0.647059  0.705882  0.0   
       25         0.0  0.0  0.0   1.941176  0.0  0.0  0.735294  0.823529  0.0   
       30         0.0  0.0  0.0   1.205882  0.0  0.0  0.735294  0.764706  0.0   
       35         0.0  0.0  0.0   0.941176  0.0  0.0  0.588235  0.617647  0.0   
       [0.1, 30]  0.0  0.0  0.0   1.264706  0.0  0.0  0.676471  0.705882  0.0   
       [0.5, 10]  0.0  0.0  0.0   6.323529  0.0  0.0  0.117647  0.411765  0.0   
       [0.5, 12]  0.0  0.0  0.0   4.205882  0.0  0.0  0.352941  0.441176  0.0   
       [0.5, 15]  0.0  0.0  0.0   3.588235  0.0  0.0  0.382353  0.411765  0.0   
       [0.5, 20]  0.0  0.0  0.0   2.558824  0.0  0.0  0.382353  0.441176  0.0   
       [0.5, 25]  0.0  0.0  0.0   1.823529  0.0  0.0  0.441176  0.500000  0.0   
       [0.5, 30]  0.0  0.0  0.0   1.264706  0.0  0.0  0.441176  0.500000  0.0   
       [0.5, 35]  0.0  0.0  0.0   1.088235  0.0  0.0  0.411765  0.441176  0.0   
cheby2 10         0.0  0.0  0.0  10.823529  0.0  0.0  0.058824  0.176471  0.0   
       15         0.0  0.0  0.0   3.882353  0.0  0.0  0.382353  0.500000  0.0   
       20         0.0  0.0  0.0   3.441176  0.0  0.0  0.558824  0.588235  0.0   
       25         0.0  0.0  0.0   2.382353  0.0  0.0  0.558824  0.617647  0.0   
       30         0.0  0.0  0.0   1.911765  0.0  0.0  0.676471  0.735294  0.0   
       35         0.0  0.0  0.0   1.205882  0.0  0.0  0.705882  0.735294  0.0   
       [0.1, 30]  0.0  0.0  0.0   1.911765  0.0  0.0  0.676471  0.735294  0.0   
       [0.5, 10]  0.0  0.0  0.0  10.205882  0.0  0.0  0.000000  0.176471  0.0   
       [0.5, 12]  0.0  0.0  0.0   7.147059  0.0  0.0  0.235294  0.470588  0.0   
       [0.5, 15]  0.0  0.0  0.0   3.676471  0.0  0.0  0.441176  0.500000  0.0   
       [0.5, 20]  0.0  0.0  0.0   3.323529  0.0  0.0  0.382353  0.411765  0.0   
       [0.5, 25]  0.0  0.0  0.0   2.411765  0.0  0.0  0.352941  0.441176  0.0   
       [0.5, 30]  0.0  0.0  0.0   1.941176  0.0  0.0  0.411765  0.500000  0.0   
       [0.5, 35]  0.0  0.0  0.0   1.235294  0.0  0.0  0.529412  0.588235  0.0   

                    b         c         d    e         f   p1        p2  
design wn                                                                
bessel 10         0.0  0.117647  0.117647  0.0  0.117647  0.0  0.117647  
       15         0.0  0.235294  0.235294  0.0  0.235294  0.0  0.235294  
       20         0.0  0.470588  0.470588  0.0  0.470588  0.0  0.470588  
       25         0.0  0.500000  0.500000  0.0  0.500000  0.0  0.500000  
       30 

In [11]:
with pd.option_context('display.precision', 2):
    display(df.groupby(level=['design', 'wn']).mean())

on   sp   dn     dp  off    u     v     w    a    b     c  \
design wn                                                                      
bessel 10         0.0  0.0  0.0  17.74  0.0  0.0  0.26  4.71  0.0  0.0  0.12   
       15         0.0  0.0  0.0   6.35  0.0  0.0  0.21  0.59  0.0  0.0  0.24   
       20         0.0  0.0  0.0   3.82  0.0  0.0  0.26  0.41  0.0  0.0  0.47   
       25         0.0  0.0  0.0   2.74  0.0  0.0  0.38  0.44  0.0  0.0  0.50   
       30         0.0  0.0  0.0   1.91  0.0  0.0  0.50  0.56  0.0  0.0  0.47   
       35         0.0  0.0  0.0   1.50  0.0  0.0  0.56  0.65  0.0  0.0  0.47   
       [0.1, 30]  0.0  0.0  0.0   2.15  0.0  0.0  0.44  0.53  0.0  0.0  0.41   
       [0.5, 10]  0.0  0.0  0.0   9.18  0.0  0.0  0.00  0.47  0.0  0.0  0.00   
       [0.5, 12]  0.0  0.0  0.0   6.18  0.0  0.0  0.00  0.21  0.0  0.0  0.00   
       [0.5, 15]  0.0  0.0  0.0   4.32  0.0  0.0  0.09  0.12  0.0  0.0  0.21   
       [0.5, 20]  0.0  0.0  0.0   3.41  0.0  0.0  0.35  0.35  0.0  0.0  0.62   
       [0.5, 25]  0.0  0.0  0.0   2.41  0.0  0.0  0.74  0.74  0.0  0.0  0.85   
       [0.5, 30]  0.0  0.0  0.0   1.82  0.0  0.0  0.88  0.88  0.0  0.0  0.88   
       [0.5, 35]  0.0  0.0  0.0   1.44  0.0  0.0  1.03  1.03  0.0  0.0  0.82   
butter 10         0.0  0.0  0.0   6.44  0.0  0.0  0.12  0.32  0.0  0.0  0.38   
       15         0.0  0.0  0.0   3.53  0.0  0.0  0.50  0.53  0.0  0.0  0.65   
       20         0.0  0.0  0.0   2.32  0.0  0.0  0.65  0.71  0.0  0.0  0.47   
       25         0.0  0.0  0.0   1.94  0.0  0.0  0.74  0.82  0.0  0.0  0.44   
       30         0.0  0.0  0.0   1.21  0.0  0.0  0.74  0.76  0.0  0.0  0.47   
       35         0.0  0.0  0.0   0.94  0.0  0.0  0.59  0.62  0.0  0.0  0.41   
       [0.1, 30]  0.0  0.0  0.0   1.26  0.0  0.0  0.68  0.71  0.0  0.0  0.38   
       [0.5, 10]  0.0  0.0  0.0   6.32  0.0  0.0  0.12  0.41  0.0  0.0  0.32   
       [0.5, 12]  0.0  0.0  0.0   4.21  0.0  0.0  0.35  0.44  0.0  0.0  0.62   
       [0.5, 15]  0.0  0.0  0.0   3.59  0.0  0.0  0.38  0.41  0.0  0.0  0.53   
       [0.5, 20]  0.0  0.0  0.0   2.56  0.0  0.0  0.38  0.44  0.0  0.0  0.38   
       [0.5, 25]  0.0  0.0  0.0   1.82  0.0  0.0  0.44  0.50  0.0  0.0  0.50   
       [0.5, 30]  0.0  0.0  0.0   1.26  0.0  0.0  0.44  0.50  0.0  0.0  0.50   
       [0.5, 35]  0.0  0.0  0.0   1.09  0.0  0.0  0.41  0.44  0.0  0.0  0.38   
cheby2 10         0.0  0.0  0.0  10.82  0.0  0.0  0.06  0.18  0.0  0.0  0.09   
       15         0.0  0.0  0.0   3.88  0.0  0.0  0.38  0.50  0.0  0.0  0.71   
       20         0.0  0.0  0.0   3.44  0.0  0.0  0.56  0.59  0.0  0.0  0.59   
       25         0.0  0.0  0.0   2.38  0.0  0.0  0.56  0.62  0.0  0.0  0.44   
       30         0.0  0.0  0.0   1.91  0.0  0.0  0.68  0.74  0.0  0.0  0.47   
       35         0.0  0.0  0.0   1.21  0.0  0.0  0.71  0.74  0.0  0.0  0.50   
       [0.1, 30]  0.0  0.0  0.0   1.91  0.0  0.0  0.68  0.74  0.0  0.0  0.44   
       [0.5, 10]  0.0  0.0  0.0  10.21  0.0  0.0  0.00  0.18  0.0  0.0  0.09   
       [0.5, 12]  0.0  0.0  0.0   7.15  0.0  0.0  0.24  0.47  0.0  0.0  0.38   
       [0.5, 15]  0.0  0.0  0.0   3.68  0.0  0.0  0.44  0.50  0.0  0.0  0.68   
       [0.5, 20]  0.0  0.0  0.0   3.32  0.0  0.0  0.38  0.41  0.0  0.0  0.59   
       [0.5, 25]  0.0  0.0  0.0   2.41  0.0  0.0  0.35  0.44  0.0  0.0  0.56   
       [0.5, 30]  0.0  0.0  0.0   1.94  0.0  0.0  0.41  0.50  0.0  0.0  0.56   
       [0.5, 35]  0.0  0.0  0.0   1.24  0.0  0.0  0.53  0.59  0.0  0.0  0.50   

                     d    e     f   p1    p2  
design wn                                     
bessel 10         0.12  0.0  0.12  0.0  0.12  
       15         0.24  0.0  0.24  0.0  0.24  
       20         0.47  0.0  0.47  0.0  0.47  
       25         0.50  0.0  0.50  0.0  0.50  
       30         0.47  0.0  0.47  0.0  0.47  
       35         0.47  0.0  0.47  0.0  0.47  
       [0.1, 30]  0.41  0.0  0.41  0.0  0.41  
       [0.5, 10]  0.00  0.0  0.00  0.0  0.00  
       [0.5, 12]  0.00  0.0 

In [ ]:
print(df.groupby(level=['design', 'wn']).mean().to_latex(float_format='%.1f'))

\begin{tabular}{llrrrrrrrrrrrrrrrr}
\toprule
 &  & on & sp & dn & dp & off & u & v & w & a & b & c & d & e & f & p1 & p2 \\
design & wn &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{14}{*}{bessel} & 10 & 0.0 & 0.0 & 0.0 & 17.7 & 0.0 & 0.0 & 0.3 & 4.7 & 0.0 & 0.0 & 0.1 & 0.1 & 0.0 & 0.1 & 0.0 & 0.1 \\
 & 15 & 0.0 & 0.0 & 0.0 & 6.4 & 0.0 & 0.0 & 0.2 & 0.6 & 0.0 & 0.0 & 0.2 & 0.2 & 0.0 & 0.2 & 0.0 & 0.2 \\
 & 20 & 0.0 & 0.0 & 0.0 & 3.8 & 0.0 & 0.0 & 0.3 & 0.4 & 0.0 & 0.0 & 0.5 & 0.5 & 0.0 & 0.5 & 0.0 & 0.5 \\
 & 25 & 0.0 & 0.0 & 0.0 & 2.7 & 0.0 & 0.0 & 0.4 & 0.4 & 0.0 & 0.0 & 0.5 & 0.5 & 0.0 & 0.5 & 0.0 & 0.5 \\
 & 30 & 0.0 & 0.0 & 0.0 & 1.9 & 0.0 & 0.0 & 0.5 & 0.6 & 0.0 & 0.0 & 0.5 & 0.5 & 0.0 & 0.5 & 0.0 & 0.5 \\
 & 35 & 0.0 & 0.0 & 0.0 & 1.5 & 0.0 & 0.0 & 0.6 & 0.6 & 0.0 & 0.0 & 0.5 & 0.5 & 0.0 & 0.5 & 0.0 & 0.5 \\
 & [0.1, 30] & 0.0 & 0.0 & 0.0 & 2.1 & 0.0 & 0.0 & 0.4 & 0.5 & 0.0 & 0.0 & 0.4 & 0.4 & 0.0 & 0.4 & 0.0 & 0.4 \\
 & [0.5, 10] & 0.0 & 0.0 & 0.0 & 9

In [17]:
print(df.groupby(level=['design', 'wn']).mean().round().astype(int).to_latex())

\begin{tabular}{llrrrrrrrrrrrrrrrr}
\toprule
 &  & on & sp & dn & dp & off & u & v & w & a & b & c & d & e & f & p1 & p2 \\
design & wn &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  &  \\
\midrule
\multirow[t]{14}{*}{bessel} & 10 & 0 & 0 & 0 & 18 & 0 & 0 & 0 & 5 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & 15 & 0 & 0 & 0 & 6 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & 20 & 0 & 0 & 0 & 4 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & 25 & 0 & 0 & 0 & 3 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & 30 & 0 & 0 & 0 & 2 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & 35 & 0 & 0 & 0 & 2 & 0 & 0 & 1 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & [0.1, 30] & 0 & 0 & 0 & 2 & 0 & 0 & 0 & 1 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & [0.5, 10] & 0 & 0 & 0 & 9 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & [0.5, 12] & 0 & 0 & 0 & 6 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & [0.5, 15] & 0 & 0 & 0 & 4 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 & 0 \\
 & [0.5, 20] & 0 &

In [23]:
df.groupby(level=['design', 'wn']).sum()

on  sp  dn   dp  off  u   v    w  a  b   c   d  e   f  p1  \
design wn                                                                     
bessel 10          0   0   0  603    0  0   9  160  0  0   4   4  0   4   0   
       15          0   0   0  216    0  0   7   20  0  0   8   8  0   8   0   
       20          0   0   0  130    0  0   9   14  0  0  16  16  0  16   0   
       25          0   0   0   93    0  0  13   15  0  0  17  17  0  17   0   
       30          0   0   0   65    0  0  17   19  0  0  16  16  0  16   0   
       35          0   0   0   51    0  0  19   22  0  0  16  16  0  16   0   
       [0.1, 30]   0   0   0   73    0  0  15   18  0  0  14  14  0  14   0   
       [0.5, 10]   0   0   0  312    0  0   0   16  0  0   0   0  0   0   0   
       [0.5, 12]   0   0   0  210    0  0   0    7  0  0   0   0  0   0   0   
       [0.5, 15]   0   0   0  147    0  0   3    4  0  0   7   7  0   7   0   
       [0.5, 20]   0   0   0  116    0  0  12   12  0  0  21  21  0  21   0   
       [0.5, 25]   0   0   0   82    0  0  25   25  0  0  29  29  0  29   0   
       [0.5, 30]   0   0   0   62    0  0  30   30  0  0  30  30  0  30   0   
       [0.5, 35]   0   0   0   49    0  0  35   35  0  0  28  28  0  28   0   
butter 10          0   0   0  219    0  0   4   11  0  0  13  13  0  13   0   
       15          0   0   0  120    0  0  17   18  0  0  22  22  0  22   0   
       20          0   0   0   79    0  0  22   24  0  0  16  16  0  16   0   
       25          0   0   0   66    0  0  25   28  0  0  15  15  0  15   0   
       30          0   0   0   41    0  0  25   26  0  0  16  16  0  16   0   
       35          0   0   0   32    0  0  20   21  0  0  14  14  0  14   0   
       [0.1, 30]   0   0   0   43    0  0  23   24  0  0  13  13  0  13   0   
       [0.5, 10]   0   0   0  215    0  0   4   14  0  0  11  11  0  11   0   
       [0.5, 12]   0   0   0  143    0  0  12   15  0  0  21  21  0  21   0   
       [0.5, 15]   0   0   0  122    0  0  13   14  0  0  18  18  0  18   0   
       [0.5, 20]   0   0   0   87    0  0  13   15  0  0  13  13  0  13   0   
       [0.5, 25]   0   0   0   62    0  0  15   17  0  0  17  17  0  17   0   
       [0.5, 30]   0   0   0   43    0  0  15   17  0  0  17  17  0  17   0   
       [0.5, 35]   0   0   0   37    0  0  14   15  0  0  13  13  0  13   0   
cheby2 10          0   0   0  368    0  0   2    6  0  0   3   3  0   3   0   
       15          0   0   0  132    0  0  13   17  0  0  24  24  0  24   0   
       20          0   0   0  117    0  0  19   20  0  0  20  20  0  20   0   
       25          0   0   0   81    0  0  19   21  0  0  15  15  0  15   0   
       30          0   0   0   65    0  0  23   25  0  0  16  16  0  16   0   
       35          0   0   0   41    0  0  24   25  0  0  17  17  0  17   0   
       [0.1, 30]   0   0   0   65    0  0  23   25  0  0  15  15  0  15   0   
       [0.5, 10]   0   0   0  347    0  0   0    6  0  0   3   3  0   3   0   
       [0.5, 12]   0   0   0  243    0  0   8   16  0  0  13  13  0  13   0   
       [0.5, 15]   0   0   0  125    0  0  15   17  0  0  23  23  0  23   0   
       [0.5, 20]   0   0   0  113    0  0  13   14  0  0  20  20  0  20   0   
       [0.5, 25]   0   0   0   82    0  0  12   15  0  0  19  19  0  19   0   
       [0.5, 30]   0   0   0   66    0  0  14   17  0  0  19  19  0  19   0   
       [0.5, 35]   0   0   0   42    0  0  18   20  0  0  17  17  0  17   0   

                  p2  
design wn             
bessel 10          4  
       15          8  
       20         16  
       25         17  
       30         16  
       35         16  
       [0.1, 30]  14  
       [0.5, 10]   0  
       [0.5, 12]   0  
       [0.5, 15]   7  
       [0.5, 20]  21  
       [0.5, 25]  29  
       [0.5, 30]  30  
       [0.5, 35]  28  
butter 10         13  
       15         22  
       20         16  
       25         15  
       30         16  
       35         14  
       [0.1, 30]  13  
       [0.5,

In [33]:
with pd.option_context('display.max_rows', 500):
    display('subject_id')
    display(df.sort_index(level='subject_id'))
    display('design')
    display(df.sort_index(level='design'))
    display('wn')
    display(df.sort_index(level='wn'))

'subject_id'

on  sp  dn  dp  off  u  v  w  a  b  c  \
category subject_id design wn                                                 
diab     8061       bessel 10          0   0   0   0    0  0  0  0  0  0  0   
                           15          0   0   0   0    0  0  0  0  0  0  0   
                           20          0   0   0   0    0  0  0  0  0  0  0   
                           25          0   0   0   0    0  0  0  0  0  0  0   
                           30          0   0   0   0    0  0  0  0  0  0  0   
...                                   ..  ..  ..  ..  ... .. .. .. .. .. ..   
         98484      cheby2 [0.5, 15]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 20]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 25]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 30]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 35]   0   0   0   1    0  0  0  0  0  0  0   

                                      d  e  f  p1  p2  
category subject_id design wn                          
diab     8061       bessel 10         0  0  0   0   0  
                           15         0  0  0   0   0  
                           20         0  0  0   0   0  
                           25         0  0  0   0   0  
                           30         0  0  0   0   0  
...                                  .. .. ..  ..  ..  
         98484      cheby2 [0.5, 15]  0  0  0   0   0  
                           [0.5, 20]  0  0  0   0   0  
                           [0.5, 25]  0  0  0   0   0  
                           [0.5, 30]  0  0  0   0   0  
                           [0.5, 35]  0  0  0   0   0  

[1428 rows x 16 columns]

'design'

on  sp  dn  dp  off  u  v  w  a  b  c  \
category subject_id design wn                                                 
control  48480      bessel 10          0   0   0  12    0  0  0  0  0  0  4   
                           15          0   0   0  12    0  0  0  0  0  0  4   
                           20          0   0   0  12    0  0  0  0  0  0  4   
                           25          0   0   0  12    0  0  0  0  0  0  4   
                           30          0   0   0  12    0  0  0  0  0  0  4   
...                                   ..  ..  ..  ..  ... .. .. .. .. .. ..   
diab     98484      cheby2 [0.5, 15]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 20]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 25]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 30]   0   0   0   1    0  0  0  0  0  0  0   
                           [0.5, 35]   0   0   0   1    0  0  0  0  0  0  0   

                                      d  e  f  p1  p2  
category subject_id design wn                          
control  48480      bessel 10         4  0  4   0   4  
                           15         4  0  4   0   4  
                           20         4  0  4   0   4  
                           25         4  0  4   0   4  
                           30         4  0  4   0   4  
...                                  .. .. ..  ..  ..  
diab     98484      cheby2 [0.5, 15]  0  0  0   0   0  
                           [0.5, 20]  0  0  0   0   0  
                           [0.5, 25]  0  0  0   0   0  
                           [0.5, 30]  0  0  0   0   0  
                           [0.5, 35]  0  0  0   0   0  

[1428 rows x 16 columns]

'wn'

on  sp  dn  dp  off  u  v  w  a  b  c  \
category subject_id design wn                                                 
control  48480      bessel 10          0   0   0  12    0  0  0  0  0  0  4   
                    butter 10          0   0   0  12    0  0  0  0  0  0  4   
                    cheby2 10          0   0   0  12    0  0  0  0  0  0  4   
         63646      bessel 10          0   0   0   0    0  0  0  0  0  0  0   
                    butter 10          0   0   0   0    0  0  0  0  0  0  0   
...                                   ..  ..  ..  ..  ... .. .. .. .. .. ..   
diab     90902      butter [0.5, 35]   0   0   0   0    0  0  0  0  0  0  0   
                    cheby2 [0.5, 35]   0   0   0   0    0  0  0  0  0  0  0   
         98484      bessel [0.5, 35]   0   0   0   1    0  0  0  0  0  0  0   
                    butter [0.5, 35]   0   0   0   1    0  0  0  0  0  0  0   
                    cheby2 [0.5, 35]   0   0   0   1    0  0  0  0  0  0  0   

                                      d  e  f  p1  p2  
category subject_id design wn                          
control  48480      bessel 10         4  0  4   0   4  
                    butter 10         4  0  4   0   4  
                    cheby2 10         4  0  4   0   4  
         63646      bessel 10         0  0  0   0   0  
                    butter 10         0  0  0   0   0  
...                                  .. .. ..  ..  ..  
diab     90902      butter [0.5, 35]  0  0  0   0   0  
                    cheby2 [0.5, 35]  0  0  0   0   0  
         98484      bessel [0.5, 35]  0  0  0   0   0  
                    butter [0.5, 35]  0  0  0   0   0  
                    cheby2 [0.5, 35]  0  0  0   0   0  

[1428 rows x 16 columns]

In [ ]:
with pd.option_context('display.max_rows', 500):
    display('sid')
    display(df.sort_index(level='sid'))
    display('design')
    display(df.sort_index(level='design'))
    display('wn')
    display(df.sort_index(level='wn'))

'sid'

on  sp  dn   dp  off  u  v    w  a  b  c  d  e  f  p1  \
sid   design wn                                                                 
40435 bessel 10          0   0   0    2    0  0  0    1  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
60747 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0

'design'

on  sp  dn   dp  off  u  v    w  a  b  c  d  e  f  p1  \
sid   design wn                                                                 
40435 bessel 10          0   0   0    2    0  0  0    1  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
60747 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
61642 bessel 10          0   0   0  124    0  0  0   73  0  0  0  0  0  0   0   
             15          0   0   0    8    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0   14    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0   76    0  0  0    0  0  0  0  0  0  0   0   
64485 bessel 10          0   0   0  122    0  0  0   54  0  0  0  0  0  0   0   
             15          0   0   0   27    0  0  0    4  0  0  0  0  0  0   0   
             20          0   0   0    3    0  0  0    1  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  1  1  0  1   0   
             [0.5, 10]   0   0   0   84    0  0  0   31  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0   36    0  0  0   12  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0   13    0  0  0    4  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0  178    0  0  0  131  0  0  0  0  0  0   0   
8061  bessel 10          0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             30          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 10]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 12]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 15]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 20]   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             [0.5, 8]    0   0   0   10    0  0  0    0  0  0  0  0  0  0   0   
82245 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
             20          0   0   0    0    0  0  0    0

'wn'

on  sp  dn   dp  off  u  v    w  a  b  c  d  e  f  p1  \
sid   design wn                                                                 
40435 bessel 10          0   0   0    2    0  0  0    1  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
60747 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
61642 bessel 10          0   0   0  124    0  0  0   73  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
64485 bessel 10          0   0   0  122    0  0  0   54  0  0  0  0  0  0   0   
      butter 10          0   0   0   23    0  0  0    3  0  0  0  0  0  0   0   
      cheby2 10          0   0   0   45    0  0  0    3  0  0  0  0  0  0   0   
8061  bessel 10          0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
82245 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  2  2  0  2   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
83263 bessel 10          0   0   0  285    0  0  0    1  0  0  0  0  0  0   0   
      butter 10          0   0   0    6    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0   26    0  0  0    0  0  0  0  0  0  0   0   
89782 bessel 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
96350 bessel 10          0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
      butter 10          0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    9    0  0  0    0  0  0  0  0  0  0   0   
98484 bessel 10          0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
      butter 10          0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 10          0   0   0    9    0  0  0    0  0  0  0  0  0  0   0   
40435 bessel 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
60747 bessel 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
61642 bessel 15          0   0   0    8    0  0  0    0  0  0  0  0  0  0   0   
      butter 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
64485 bessel 15          0   0   0   27    0  0  0    4  0  0  0  0  0  0   0   
      butter 15          0   0   0    1    0  0  0    0  0  0  1  1  0  1   0   
      cheby2 15          0   0   0   14    0  0  0    2  0  0  1  1  0  1   0   
8061  bessel 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      cheby2 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
82245 bessel 15          0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
      butter 15          0   0   0    0    0  0  0    0  0  0  4  4  0  4   0   
      cheby2 15          0   0   0    0    0  0  0    0

In [22]:
with pd.option_context('display.max_rows', 500):
    display(df.swaplevel(0, 1).swaplevel(1, 2).sort_index(level=['design', 'wn']))

on  sp  dn   dp  off  u  v    w  a  b  c  d  e  f  p1  \
design wn        sid                                                            
bessel 10        40435   0   0   0    2    0  0  0    1  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0  124    0  0  0   73  0  0  0  0  0  0   0   
                 64485   0   0   0  122    0  0  0   54  0  0  0  0  0  0   0   
                 8061    0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0  285    0  0  0    1  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
                 98484   0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
       15        40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    8    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0   27    0  0  0    4  0  0  0  0  0  0   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0   20    0  0  0    6  0  0  0  0  0  0   0   
                 98484   0   0   0   20    0  0  0    6  0  0  0  0  0  0   0   
       20        40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0    3    0  0  0    1  0  0  0  0  0  0   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0    6    0  0  0    0  0  0  0  0  0  0   0   
                 98484   0   0   0    6    0  0  0    0  0  0  0  0  0  0   0   
       30        40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0    0    0  0  0    0  0  0  1  1  0  1   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  1    1  0  0  1  1  0  1   0   
                 83263   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0    3    0  0  0    0  0  0  0  0  0  0   0   
                 98484   0   0   0    3    0  0  0    0  0  0  0  0  0  0   0   
       [0.5, 10] 40435   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0   14    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0   84    0  0  0   31  0  0  0  0  0  0   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0

In [24]:
with pd.option_context('display.max_rows', 500):
    display(df.swaplevel(0, 2).sort_index(level=['wn', 'design']))

on  sp  dn   dp  off  u  v    w  a  b  c  d  e  f  p1  \
wn        design sid                                                            
10        bessel 40435   0   0   0    2    0  0  0    1  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0  124    0  0  0   73  0  0  0  0  0  0   0   
                 64485   0   0   0  122    0  0  0   54  0  0  0  0  0  0   0   
                 8061    0   0   0    2    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0  285    0  0  0    1  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
                 98484   0   0   0  101    0  0  0   75  0  0  0  0  0  0   0   
          butter 40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0   23    0  0  0    3  0  0  0  0  0  0   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  2  2  0  2   0   
                 83263   0   0   0    6    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
                 98484   0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
          cheby2 40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0   45    0  0  0    3  0  0  0  0  0  0   0   
                 8061    0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0   26    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0    9    0  0  0    0  0  0  0  0  0  0   0   
                 98484   0   0   0    9    0  0  0    0  0  0  0  0  0  0   0   
15        bessel 40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    8    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0   27    0  0  0    4  0  0  0  0  0  0   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 83263   0   0   0   12    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 96350   0   0   0   20    0  0  0    6  0  0  0  0  0  0   0   
                 98484   0   0   0   20    0  0  0    6  0  0  0  0  0  0   0   
          butter 40435   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 60747   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 61642   0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 64485   0   0   0    1    0  0  0    0  0  0  1  1  0  1   0   
                 8061    0   0   0    0    0  0  0    0  0  0  0  0  0  0   0   
                 82245   0   0   0    0    0  0  0    0  0  0  4  4  0  4   0   
                 83263   0   0   0    1    0  0  0    0  0  0  0  0  0  0   0   
                 89782   0   0   0    0    0  0  0    0